# Speech Denoising — EDA & Preprocessing

Exploratory data analysis and preprocessing for the VoiceBank+DEMAND dataset.

**Input:** noisy speech audio | **Output:** clean speech audio

---

## 1. Environment

In [ ]:
!python --version


In [ ]:
import numpy as np
import librosa as lb
import matplotlib.pyplot as plt
import soundfile as sf
from pathlib import Path
from IPython.display import Audio
from tqdm import tqdm

## 2. Dataset Paths

In [ ]:
noisy_train_dir = Path('data/archive/noisy_trainset_28spk_wav')
clean_train_dir = Path('data/archive/clean_trainset_28spk_wav')
noisy_test_dir = Path('data/archive/noisy_testset_wav')
clean_test_dir = Path('data/archive/clean_testset_wav')

## 3. EDA — Trainset

### 3.1 File Count Validation

Verify that noisy and clean trainsets have equal number of files (required for supervised learning).

In [ ]:
noisy_count = len(list(noisy_train_dir.glob('*.wav')))
print(noisy_count)

In [ ]:
clean_count = len(list(clean_train_dir.glob('*.wav')))
print(clean_count)

### 3.2 File Listing & Pair Validation

Sort files alphabetically to ensure noisy/clean pairs align by index.

In [ ]:
noisy_files = sorted(noisy_train_dir.glob('*.wav'))
print(noisy_files[1])

In [ ]:
clean_files = sorted(clean_train_dir.glob('*.wav'))
print(clean_files[1])

### 3.3 Audio Loading

Load sample files with `sr=None` to preserve original sample rate (16000 Hz).

In [ ]:
noisy_sample, sr_train = lb.load(noisy_files[0], sr=None)
print(noisy_sample)
print(sr_train)


In [ ]:
clean_sample, _ = lb.load(clean_files[0], sr=None)
print(clean_sample)


### 3.4 Listening Test

Verify noisy/clean pair content matches by ear using IPython Audio.

In [ ]:
Audio(noisy_files[0])


### 3.5 Length Check

Confirm that noisy and clean versions of the same utterance have identical sample counts.

In [ ]:
file_clean, _ = lb.load(clean_files[0], sr=None)
file_noisy, _ = lb.load(noisy_files[0], sr=None)

if len(file_clean) == len(file_noisy):
    print("same size")
else:
    print("the size is not the same")


### 3.6 Spectrogram Visualization

STFT spectrogram comparison (log frequency scale) — clean (top) vs noisy (bottom).
Noise is visible as uniform energy across all frequencies, especially in silent regions.

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(2,1,1)
hop_length = 64
D_clean = lb.amplitude_to_db(np.abs(lb.stft(file_clean, hop_length=hop_length)))
lb.display.specshow(D_clean, sr=16000, x_axis='time', y_axis='log', hop_length=hop_length)
plt.title("Clean")
plt.subplot(2,1,2)
D_noisy = lb.amplitude_to_db(np.abs(lb.stft(file_noisy, hop_length=hop_length)))
lb.display.specshow(D_noisy, sr=16000, x_axis='time', y_axis='log', hop_length=hop_length)
plt.title("Noisy")
plt.tight_layout()


### 3.7 Duration Analysis

Check min/max file duration across entire trainset to inform chunking strategy.

In [ ]:
train_lengths = []
for file in tqdm(noisy_files, total=len(noisy_files)):
    y, sr = lb.load(file, sr=None)
    train_lengths.append(len(y))

print(f"max: {max(train_lengths)/16000:.2f}s, min: {min(train_lengths)/16000:.2f}s")


## 4. Preprocessing — Trainset

### 4.1 Chunking

Slice all audio files into fixed 1-second chunks (16000 samples).
Last incomplete chunk is dropped. Both noisy and clean are chunked identically to preserve pairing.

In [ ]:
noisy_chunks = []
for file in tqdm(noisy_files, total=len(noisy_files)):
    y, sr = lb.load(file, sr=None)
    y = y[:(len(y)//sr) * sr]
    C = np.split(y, len(y)//sr)
    noisy_chunks.append(C)

In [ ]:
len(noisy_chunks[0][0])

In [ ]:
clean_chunks = []
for file in tqdm(clean_files, total=len(clean_files)):
    y, sr = lb.load(file, sr=None)
    y = y[:(len(y)//sr) * sr]
    C = np.split(y, len(y)//sr)
    clean_chunks.append(C)

In [ ]:
len(clean_chunks[0][0])

### 4.2 Flatten to List

Convert nested list `[file][chunk]` to flat list of arrays for dataset construction.

In [ ]:
train_clean_chunks = []
for file_chunks in tqdm(clean_chunks, total=len(clean_chunks)):
    for chunk in file_chunks:
        train_clean_chunks.append(chunk)


In [ ]:
train_clean_chunks[0]

In [ ]:
train_noisy_chunks = []
for file_chunks in tqdm(noisy_chunks, total=len(noisy_chunks)):
    for chunk in file_chunks:
        train_noisy_chunks.append(chunk)


In [ ]:
train_noisy_chunks[0]

In [ ]:
len(train_noisy_chunks) == len(train_clean_chunks)

### 4.3 Save to Disk

Save preprocessed arrays as `.npy` files to avoid reprocessing on every training run.

In [ ]:
np.save('data/noisy_chunks.npy', train_noisy_chunks)
np.save('data/clean_chunks.npy',train_clean_chunks)

## 5. EDA & Preprocessing — Testset

### 5.1 File Count Validation

In [ ]:
noisy_test_count = len(list(noisy_test_dir.glob('*.wav')))
print(noisy_test_count)

In [ ]:
clean_test_count = len(list(clean_test_dir.glob('*.wav')))
print(clean_test_count)

### 5.2 File Listing

In [ ]:
noisy_test_files = sorted(noisy_test_dir.glob('*.wav'))
print(noisy_test_files[1])

In [ ]:
clean_test_files = sorted(clean_test_dir.glob('*.wav'))
print(clean_test_files[1])

### 5.3 Pair Validation

In [ ]:
noisy_test_sample, sr_test = lb.load(noisy_test_files[0], sr=None)
print(noisy_test_sample)
print(sr_test)


In [ ]:
file_test_clean, _ = lb.load(clean_test_files[0], sr=None)
file_test_noisy, _ = lb.load(noisy_test_files[0], sr=None)

if len(file_test_clean) == len(file_test_noisy):
    print("same size")
else:
    print("the size is not the same")


### 5.4 Duration Analysis

In [ ]:
test_lengths = []
for file in tqdm(noisy_test_files, total=len(noisy_test_files)):
    y, sr = lb.load(file, sr=None)
    test_lengths.append(len(y))

print(f"max: {max(test_lengths)/16000:.2f}s, min: {min(test_lengths)/16000:.2f}s")


### 5.5 Chunking

In [ ]:
noisy_test_chunks = []
for file in tqdm(noisy_test_files, total=len(noisy_test_files)):
    y, sr = lb.load(file, sr=None)
    y = y[:(len(y)//sr) * sr]
    C = np.split(y, len(y)//sr)
    noisy_test_chunks.append(C)

In [ ]:
len(noisy_test_chunks[0][0])

In [ ]:
clean_test_chunks = []
for file in tqdm(clean_test_files, total=len(clean_test_files)):
    y, sr = lb.load(file, sr=None)
    y = y[:(len(y)//sr) * sr]
    C = np.split(y, len(y)//sr)
    clean_test_chunks.append(C)

In [ ]:
len(clean_test_chunks[0][0])

In [ ]:
len(noisy_test_chunks) == len(clean_test_chunks)

### 5.6 Flatten to List

In [ ]:
test_clean_chunks = []
for file_chunks in tqdm(clean_test_chunks, total=len(clean_test_chunks)):
    for chunk in file_chunks:
        test_clean_chunks.append(chunk)


In [ ]:
test_clean_chunks[0]

In [ ]:
test_noisy_chunks = []
for file_chunks in tqdm(noisy_test_chunks, total=len(noisy_test_chunks)):
    for chunk in file_chunks:
        test_noisy_chunks.append(chunk)


In [ ]:
test_noisy_chunks[0]

In [ ]:
len(test_noisy_chunks) == len(test_clean_chunks)

### 5.7 Save to Disk

In [ ]:
np.save('data/noisy_chunks_test.npy', test_noisy_chunks)
np.save('data/clean_chunks_test.npy',test_clean_chunks)